# 02 Bronze

Auto Loader lee `landing/` con el esquema explícito de la Parte 1 y hace un `MERGE`
que solo inserta `(MMSI, BaseDateTime)` nuevos. `fecha` es el widget del job: no filtra
el stream (01 ya publicó ese día); queda en la fila de métricas.

No ejecutes este stream en dos sesiones a la vez: el checkpoint es uno solo.
`useIncrementalListing=false` lista la carpeta completa. Con un CSV por día es barato,
y un archivo nuevo no depende de quedar delante del último nombre ya visto.


In [ ]:
import datetime as dt

dbutils.widgets.text("fecha", "2023-06-08", "Fecha (YYYY-MM-DD)")
fecha = dbutils.widgets.get("fecha").strip()
try:
    fecha_dia = dt.date.fromisoformat(fecha)
except ValueError as exc:
    raise ValueError(f"fecha debe tener la forma YYYY-MM-DD. Llego: {fecha!r}") from exc

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "dbr_acad_ingsistemas_mine"
SCHEMA = "default"
VOLUME = "oceanwatch_data"

VOLUME_NAME = f"`{CATALOG}`.`{SCHEMA}`.`{VOLUME}`"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
RAW_ZIPS = f"{VOLUME_ROOT}/raw_zips"
LANDING = f"{VOLUME_ROOT}/landing"
CHECKPOINTS = f"{VOLUME_ROOT}/checkpoints"
SCHEMAS = f"{VOLUME_ROOT}/_schemas"
REF = f"{VOLUME_ROOT}/ref"
STAGING = f"{RAW_ZIPS}/_staging"

BRONZE = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_bronze_ais`"
SILVER = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_silver_ais`"
CUARENTENA = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_cuarentena`"
METRICAS = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_metricas`"
PUERTOS = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_puertos`"
GOLD_TRAY = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_gold_trayectorias`"
GOLD_PUERTO = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_gold_presencia_puertos`"
GOLD_TRAFICO = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_gold_trafico_celda_hora`"

CHECKPOINT = f"{CHECKPOINTS}/bronze_ais"
SCHEMA_LOC = f"{SCHEMAS}/bronze_ais"
print(fecha_dia.isoformat(), CHECKPOINT)


In [ ]:
from pyspark.sql import functions as F

ais_schema = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG double,
    COG double,
    Heading double,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType int,
    Status int,
    Length double,
    Width double,
    Draft double,
    Cargo string,
    TransceiverClass string
"""


def leer_landing():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOC)
        .option("cloudFiles.schemaEvolutionMode", "none")
        .option("cloudFiles.includeExistingFiles", "true")
        .option("cloudFiles.allowOverwrites", "false")
        .option("cloudFiles.useIncrementalListing", "false")
        .option("header", "true")
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
        .schema(ais_schema)
        .load(LANDING)
        .select(
            "MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading",
            "VesselName", "IMO", "CallSign", "VesselType", "Status",
            "Length", "Width", "Draft", "Cargo", "TransceiverClass",
            F.col("_metadata.file_path").alias("_source_file"),
            F.current_timestamp().alias("_ingested_at"),
        )
    )


def merge_into_bronze(batch_df, batch_id: int) -> None:
    sesion = batch_df.sparkSession
    sin_duplicados = batch_df.dropDuplicates(["MMSI", "BaseDateTime"])
    vista = f"oceanwatch_bronze_batch_{int(batch_id)}"
    sin_duplicados.createOrReplaceTempView(vista)
    sesion.sql(f"""
        MERGE INTO {BRONZE} AS destino
        USING {vista} AS origen
        ON destino.MMSI = origen.MMSI
       AND destino.BaseDateTime = origen.BaseDateTime
        WHEN NOT MATCHED THEN INSERT *
    """)
    sesion.catalog.dropTempView(vista)


def filas(tabla: str) -> int:
    return int(spark.sql(f"SELECT COUNT(*) AS n FROM {tabla}").collect()[0]["n"])


for consulta in list(spark.streams.active):
    if consulta.name == "oceanwatch_bronze_ais":
        consulta.stop()

antes = filas(BRONZE)
consulta = (
    leer_landing()
    .writeStream.foreachBatch(merge_into_bronze)
    .queryName("oceanwatch_bronze_ais")
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)
consulta.awaitTermination()
despues = filas(BRONZE)

spark.sql(f"""
INSERT INTO {METRICAS}
SELECT
  DATE '{fecha_dia.isoformat()}',
  'bronze',
  CAST({antes} AS BIGINT),
  CAST({despues} AS BIGINT),
  CAST(NULL AS BIGINT),
  CAST(NULL AS BIGINT),
  CAST(NULL AS BIGINT),
  current_timestamp()
""")
print(f"Ejecución bronze {fecha_dia.isoformat()}: antes={antes} despues={despues}")
